# Customer Churn Prediction using XGBoost

## 1.Problem Statement

- In today’s competitive market, retaining customers is more valuable than acquiring new ones. Companies often struggle to identify customers who are likely to leave (churn).

- The goal of this project is to build a machine learning model that predicts whether a customer will churn based on their behavior, subscription details, and service usage.

## 2. Objectives
- Predict customer churn using classification models
- Compare performance of:
    - Baseline Model (XGBoost before tuning)
    - Tuned XGBoost Model
    - AdaBoost Classifier
- Improve prediction accuracy using hyperparameter tuning
- Analyze model performance using:
    - Accuracy
    - Precision
    - Recall
    - F1-score

In [4]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


## 3.Import Libraries


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report


from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier

## 4.Load Dataset

In [7]:
df = pd.read_csv("adaboost_churn_dataset_v2.csv")
df.head()

,age,monthly_charge,tenure,support_calls,contract_type,internet_service,payment_method,region,device_type,churn
0,56,1612,19,6,Monthly,DSL,Credit Card,Urban,Mobile,1
1,69,777,70,2,Yearly,NaN,Credit Card,Semi-Urban,Mobile,0
2,46,1554,13,0,Monthly,DSL,UPI,Urban,Mobile,0
3,32,1222,23,3,Yearly,DSL,Cash,Urban,Mobile,1
4,60,719,1,8,Yearly,Fiber,UPI,Urban,Mobile,1


## 5.Check Missing Values & Handle

In [8]:
df.isnull().sum()

age                   0
monthly_charge        0
tenure                0
support_calls         0
contract_type         0
internet_service    130
payment_method        0
region                0
device_type           0
churn                 0
dtype: int64

In [9]:
df["internet_service"]=df["internet_service"].fillna(df["internet_service"].mode()[0], inplace=True)

C:\Users\yashk\AppData\Local\Temp\ipykernel_6176\35770522.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["internet_service"]=df["internet_service"].fillna(df["internet_service"].mode()[0], inplace=True)


In [10]:
df.isnull().sum()

age                 0
monthly_charge      0
tenure              0
support_calls       0
contract_type       0
internet_service    0
payment_method      0
region              0
device_type         0
churn               0
dtype: int64

## 6.Data Preprocessing

In [16]:
df = pd.get_dummies(df, drop_first=True)
df

,age,monthly_charge,tenure,support_calls,churn,contract_type_Yearly,internet_service_Fiber,payment_method_Cash,payment_method_Credit Card,payment_method_UPI,region_Semi-Urban,region_Urban,device_type_Mobile,device_type_Tablet
0,56,1612,19,6,1,False,False,False,True,False,False,True,True,False
1,69,777,70,2,0,True,True,False,True,False,True,False,True,False
2,46,1554,13,0,0,False,False,False,False,True,False,True,True,False
3,32,1222,23,3,1,True,False,True,False,False,False,True,True,False
4,60,719,1,8,1,True,True,False,False,True,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,59,1704,35,3,0,False,False,True,False,False,True,False,True,False
1196,69,1425,19,3,1,False,True,True,False,False,False,False,False,True
1197,32,793,8,6,1,True,False,False,False,True,True,False,True,False
1198,64,1967,26,6,1,False,True,False,False,False,False,True,True,False


## 9.Model Building

## i) Seperate Target & Feature

In [12]:
X = df.drop("churn", axis=1)
y = df["churn"]

## ii) Split The Dataset

In [13]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)


## Model 1: XGBoost (Before Tuning)

In [22]:
xgb_model = XGBClassifier(random_state=42)

xgb_model.fit(X_train, y_train)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

XGBoost Accuracy: 0.8708333333333333
              precision    recall  f1-score   support

           0       0.87      0.89      0.88       128
           1       0.87      0.85      0.86       112

    accuracy                           0.87       240
   macro avg       0.87      0.87      0.87       240
weighted avg       0.87      0.87      0.87       240



## Model 2: XGBoost (After Tuning)

In [26]:
param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.01, 0.05],
    'max_depth': [3, 5],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1],
    'gamma': [0, 0.1],
}

grid = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)





GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1], 'gamma': [0, 0.1],
                         'learning_rate': [0.01, 0.05], 'max_depth': [3, 5],
                         'n_estimators': [200, 300], 'subsample': [0.8, 1]},
             scoring='accuracy')

In [27]:

best_xgb = grid.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)



print("Tuned XGBoost Accuracy:", accuracy_score(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))

Tuned XGBoost Accuracy: 0.8583333333333333
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       128
           1       0.88      0.81      0.84       112

    accuracy                           0.86       240
   macro avg       0.86      0.86      0.86       240
weighted avg       0.86      0.86      0.86       240



## Model 3 : ADABoost Classifier

In [28]:
ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada_model.fit(X_train, y_train)



AdaBoostClassifier(learning_rate=0.1, n_estimators=100, random_state=42)

In [29]:
y_pred_ada = ada_model.predict(X_test)

print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

AdaBoost Accuracy: 0.8416666666666667
              precision    recall  f1-score   support

           0       0.81      0.91      0.86       128
           1       0.89      0.76      0.82       112

    accuracy                           0.84       240
   macro avg       0.85      0.84      0.84       240
weighted avg       0.85      0.84      0.84       240



## 10.Conclusion
- This project focused on predicting customer churn using ensemble learning techniques, specifically XGBoost and AdaBoost classifiers. The baseline XGBoost model achieved the highest accuracy of 87%, demonstrating strong predictive capability. Hyperparameter tuning improved model stability, resulting in an accuracy of 85.5%, though slightly lower than the default model due to potential overfitting or limited parameter space. AdaBoost also performed competitively with an accuracy of 84%, but showed some imbalance in class prediction. Overall, XGBoost proved to be the most effective and reliable model for this dataset, highlighting the power of gradient boosting in classification tasks.